# 04 — Resolve player team roles and phase disagreements

**Goal:** correct the team-role blind spot in Notebook 03 using SkillCorner match metadata. This is a bounded correction to the single-match canonical table, **not a new audit**. Player-to-team assignments come from the match roster; no spatial or future-outcome heuristics are used. Nine phase disagreements are preserved for targeted investigation, not silently overwritten.

**Inputs:** `outputs/P1_canonical_decisions_match_1874553.jsonl`, local `data/matches/1874553/1874553_match.json`, and matching extrapolated tracking JSONL. **Outputs:** only `outputs/P1_canonical_decisions_match_1874553_ROLES.jsonl`, `outputs/P1_team_roles_gate.json`.

In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import json, math
ROOT=next((p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'data'/'matches').is_dir()),None)
if ROOT is None: raise FileNotFoundError('Run inside your SkillCorner repository')
OUT=ROOT/'outputs';OUT.mkdir(exist_ok=True)
MID='1874553'; MATCH=ROOT/'data'/'matches'/MID
SRC=OUT/f'P1_canonical_decisions_match_{MID}.jsonl'
if not SRC.exists():raise FileNotFoundError(f'Run Notebook 03 first: {SRC}')
rows=[json.loads(line) for line in SRC.open() if line.strip()]
meta=json.loads((MATCH/f'{MID}_match.json').read_text())
print('Canonical decisions:',len(rows),'Roster players:',len(meta.get('players',[])))

Canonical decisions: 959 Roster players: 36


## 1. Identify the player key explicitly

SkillCorner match metadata includes `players[].id`, `players[].trackable_object`, and `players[].team_id`. We check which ID matches the carrier event IDs before classifying any teammate or opponent. This is a source-backed mapping, not a guessed assignment.

In [2]:
players=meta.get('players',[])
by_player_id={str(p['id']):str(p['team_id']) for p in players if p.get('id') is not None and p.get('team_id') is not None}
by_trackable={str(p['trackable_object']):str(p['team_id']) for p in players if p.get('trackable_object') is not None and p.get('team_id') is not None}
carrier_ids=[str(r['carrier_player_id']) for r in rows if r.get('carrier_player_id') is not None]
hits_id=sum(c in by_player_id for c in carrier_ids)
hits_trackable=sum(c in by_trackable for c in carrier_ids)
print('Carrier-key match: player.id',hits_id,'/',len(carrier_ids))
print('Carrier-key match: player.trackable_object',hits_trackable,'/',len(carrier_ids))
if hits_id==len(carrier_ids) and hits_trackable<len(carrier_ids):
    player_to_team=by_player_id; scheme='player.id'
elif hits_trackable==len(carrier_ids) and hits_id<len(carrier_ids):
    player_to_team=by_trackable; scheme='player.trackable_object'
elif hits_id==len(carrier_ids) and hits_trackable==len(carrier_ids):
    conflicts={x for x in carrier_ids if by_player_id.get(x)!=by_trackable.get(x)}
    if conflicts: raise AssertionError('Ambiguous identifier schemes give conflicting teams; stop and inspect')
    player_to_team=by_player_id; scheme='player.id (both maps equivalent for carriers)'
else: raise AssertionError('No roster identifier maps every carrier. Do not guess mapping.')
print('Verified roster lookup:',scheme)
print('Known team IDs:',sorted(set(player_to_team.values())))

Carrier-key match: player.id 959 / 959
Carrier-key match: player.trackable_object 0 / 959
Verified roster lookup: player.id
Known team IDs: ['1802', '866']


## 2. Reattach team roles to exact tracking frames

Only use recorded roster teams. Preserve `is_detected` for each position. Missing or unrecognized IDs remain unresolved. We additionally require every identified carrier to belong to the event's attacking team; otherwise stop before writing the revised table.

In [3]:
 def finite_xy(p):
    try:
        x, y = float(p["x"]), float(p["y"])
        return [x, y] if math.isfinite(x) and math.isfinite(y) else None
    except (TypeError, ValueError, KeyError):
        return None


wanted = {(int(r["period"]), int(r["frame"])) for r in rows}

frames = {}
skipped_missing_period = 0
skipped_invalid_keys = 0

with (MATCH / f"{MID}_tracking_extrapolated.jsonl").open() as stream:
    for line in stream:
        obj = json.loads(line)

        # Handle records with missing or invalid period/frame identifiers.
        if obj.get("period") is None or obj.get("frame") is None:
            skipped_missing_period += 1
            continue

        try:
            key = (int(obj["period"]), int(obj["frame"]))
        except (TypeError, ValueError):
            skipped_invalid_keys += 1
            continue

        if key in wanted:
            frames[key] = obj

        if len(frames) == len(wanted):
            break

if len(frames) != len(wanted):
    missing = sorted(wanted - frames.keys())
    raise AssertionError(
        f"Missing {len(missing)} tracking frames. "
        f"First missing keys: {missing[:10]}"
    )

errors = []

for r in rows:
    frame = frames[(int(r["period"]), int(r["frame"]))]

    carrier_id = str(r["carrier_player_id"])
    own_team = str(r["attacking_team_id"])

    if player_to_team.get(carrier_id) != own_team:
        errors.append({
            "frame": r["frame"],
            "carrier": carrier_id,
            "expected": own_team,
            "roster": player_to_team.get(carrier_id)
        })

    bins = {
        f"{role}_xy_{quality}": []
        for role in ("teammates", "opponents")
        for quality in ("detected", "extrapolated")
    }

    unknown = []

    for p in frame.get("player_data") or []:
        pid = p.get("player_id")
        pid = str(pid) if pid is not None else None

        if pid == carrier_id:
            continue

        point = finite_xy(p)

        if point is None:
            continue

        team = player_to_team.get(pid)

        if team is None:
            unknown.append(pid)
            continue

        role = "teammates" if team == own_team else "opponents"
        quality = (
            "detected"
            if p.get("is_detected") is True
            else "extrapolated"
        )

        bins[f"{role}_xy_{quality}"].append({
            "player_id": pid,
            "xy": point
        })

    r.update(bins)
    r["unresolved_players"] = len(unknown)
    r["unknown_player_ids"] = unknown

    r["team_roles_resolved"] = (
        not unknown
        and player_to_team.get(carrier_id) == own_team
    )

if errors:
    raise AssertionError(
        f"Carrier team inconsistency, first cases: {errors[:5]}"
    )

print("Tracking frames matched:", len(frames), "/", len(wanted))

print("Records skipped due to missing period/frame:",
      skipped_missing_period)

print("Records skipped due to invalid identifiers:",
      skipped_invalid_keys)

print(
    "Rows with fully resolved roles:",
    sum(r["team_roles_resolved"] for r in rows),
    "/",
    len(rows)
)

print(
    "Unresolved players per frame:",
    Counter(r["unresolved_players"] for r in rows)
)

Tracking frames matched: 959 / 959
Records skipped due to missing period/frame: 19
Records skipped due to invalid identifiers: 0
Rows with fully resolved roles: 959 / 959
Unresolved players per frame: Counter({0: 959})


## 3. Isolate the nine phase team disagreements

Notebook 03 used the half-open frame convention. Its `team_disagreement` flag tells us that a time-matching phase has a different attacking-team ID. These rows are *not* resolved by player mapping. We record their period/frame, adjacent phase definitions and carrier team to determine if the mismatch is a possession-transition boundary or another issue.

In [4]:
import csv
with (MATCH/f'{MID}_phases_of_play.csv').open(newline='',encoding='utf-8-sig') as f:
    phase_rows=list(csv.DictReader(f))
disagreements=[]
for r in rows:
    if r['phase_status']!='team_disagreement':continue
    t=int(r['frame']);period=int(r['period'])
    near=[]
    for p in phase_rows:
        try:
            if int(p['period'])!=period:continue
            a,b=int(p['frame_start']),int(p['frame_end'])
        except (ValueError,TypeError,KeyError):continue
        if a-2<=t<=b+2:
            near.append({'index':p.get('index'),'start':a,'end':b,'team_in_possession_id':p.get('team_in_possession_id'),'in_phase':p.get('team_in_possession_phase_type'),'out_phase':p.get('team_out_of_possession_phase_type')})
    disagreements.append({'period':period,'frame':t,'event_id':r['possession_event_id'],'carrier_id':r['carrier_player_id'],'event_attacking_team':r['attacking_team_id'],'nearby_phases':near})
print('Team disagreements:',len(disagreements))
for item in disagreements:print(json.dumps(item))

Team disagreements: 9
{"period": 1, "frame": 6732, "event_id": "8_123", "carrier_id": "26766", "event_attacking_team": "866", "nearby_phases": [{"index": "67", "start": 6682, "end": 6882, "team_in_possession_id": "1802", "in_phase": "set_play", "out_phase": "defending_set_play"}]}
{"period": 1, "frame": 6851, "event_id": "8_127", "carrier_id": "50968", "event_attacking_team": "866", "nearby_phases": [{"index": "67", "start": 6682, "end": 6882, "team_in_possession_id": "1802", "in_phase": "set_play", "out_phase": "defending_set_play"}]}
{"period": 1, "frame": 9487, "event_id": "8_156", "carrier_id": "560893", "event_attacking_team": "866", "nearby_phases": [{"index": "82", "start": 9327, "end": 9527, "team_in_possession_id": "1802", "in_phase": "set_play", "out_phase": "defending_set_play"}]}
{"period": 1, "frame": 15091, "event_id": "8_253", "carrier_id": "51005", "event_attacking_team": "866", "nearby_phases": [{"index": "127", "start": 15054, "end": 15254, "team_in_possession_id": "1

## 4. Persist compact gate and corrected single-match scaffold

We distinguish role resolution, player detection, frame lookup, phase attribution, and label leakage. A role-resolved row may still have an unresolved phase or extrapolated coordinates. Do not use the revised table in inference until the flagged cases have been considered.

In [5]:
gate={'created_utc':datetime.now(timezone.utc).isoformat(),'match_id':MID,'decisions':len(rows),
      'roster_identifier_scheme':scheme,'roster_players':len(players),'carrier_roster_hits':len(carrier_ids),
      'tracking_frames_found':len(frames),'team_roles_resolved':sum(r['team_roles_resolved'] for r in rows),
      'rows_with_unresolved_players':sum(r['unresolved_players']>0 for r in rows),
      'roster_team_ids':sorted(set(player_to_team.values())),
      'phase_status_counts':dict(Counter(r['phase_status'] for r in rows)),
      'phase_disagreement_cases':disagreements,
      'caveats':['Team role derived from match roster metadata, not guessed from positions',
                 'is_detected still distinguishes direct detections from extrapolated positions',
                 'Phase disagreements not silently relabelled',
                 'Observed carry and end_type remain outcome labels, never pre-action predictors',
                 'Only one match: no claims of cross-match validity']}
p=OUT/'P1_team_roles_gate.json';p.write_text(json.dumps({'gate':gate},indent=2))
q=OUT/f'P1_canonical_decisions_match_{MID}_ROLES.jsonl'
with q.open('w') as f:
    for row in rows:f.write(json.dumps(row,separators=(',',':'))+'\n')
print('Gate:',p,'\nCorrected scaffold:',q)
print('Upload only:',p.name)

Gate: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P1_team_roles_gate.json 
Corrected scaffold: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P1_canonical_decisions_match_1874553_ROLES.jsonl
Upload only: P1_team_roles_gate.json
